# Kyphosis Prediction — EDA and Classification
Reproducible notebook based on the submitted Experiment 10 report. The results below are from a documented split and may differ from the original report.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, roc_auc_score, RocCurveDisplay
import joblib
base = Path.cwd()
if not (base / "data/kyphosis.csv").exists(): base = base.parent
df = pd.read_csv(base / "data/kyphosis.csv")
df.head()

## 1. Dataset review and cleaning

In [ ]:
print("Shape:", df.shape)
print("Missing values:\n", df.isna().sum())
print("Class counts:\n", df.Kyphosis.value_counts())
display(df.describe())
assert df.shape == (81,4)
assert df.isna().sum().sum()==0

## 2. Exploratory data analysis

In [ ]:
sns.countplot(data=df, x="Kyphosis")
plt.title("Class balance")
plt.show()
sns.pairplot(df, hue="Kyphosis")
plt.show()

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(13,3))
for ax,col in zip(axes,["Age","Number","Start"]): sns.histplot(data=df,x=col,hue="Kyphosis",ax=ax,kde=True)
plt.tight_layout();plt.show()

## 3. Train/test split (70/30)
The dataset is imbalanced. Stratification keeps both classes represented in the held-out test set.

In [ ]:
X=df[["Age","Number","Start"]]
y=(df["Kyphosis"]=="present").astype(int)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.30,random_state=42,stratify=y)
print("Training:",len(X_train),"Testing:",len(X_test))

## 4. Model training and evaluation

In [ ]:
models={"Decision Tree":DecisionTreeClassifier(random_state=42),"Random Forest":RandomForestClassifier(n_estimators=100,random_state=42)}
results=[]
for name,model in models.items():
    model.fit(X_train,y_train)
    preds=model.predict(X_test)
    scores=model.predict_proba(X_test)[:,1]
    print("\n",name,"\n",classification_report(y_test,preds,target_names=["absent","present"],zero_division=0))
    print("ROC-AUC:",round(roc_auc_score(y_test,scores),3))
    ConfusionMatrixDisplay.from_predictions(y_test,preds,display_labels=["absent","present"])
    plt.title(name);plt.show()
    results.append({"Model":name,"Accuracy":accuracy_score(y_test,preds),"ROC-AUC":roc_auc_score(y_test,scores)})
pd.DataFrame(results)

## 5. Feature importance and model export

In [ ]:
rf=models["Random Forest"]
pd.Series(rf.feature_importances_,index=X.columns).sort_values().plot.barh(title="Random Forest feature importance")
plt.show()
(base / "models").mkdir(exist_ok=True)
joblib.dump(models["Decision Tree"],base / "models/decision_tree.joblib")
joblib.dump(models["Random Forest"],base / "models/random_forest.joblib")
print("Saved trained models")

## 6. Conclusion
This small historical dataset is for teaching. Performance on the `present` minority class must be considered alongside overall accuracy. Results depend on the random split and model parameters; do not interpret predictions as clinical diagnoses.